In [1]:
from __future__ import annotations

import re
import csv
import json
import math
import hashlib
import unicodedata
import datetime as dt
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd

In [2]:
# -------------------------------------------------------------------
# Follow-up quality CSV prep
# -------------------------------------------------------------------

FOLLOWUP_DATA_ROOT = Path("ai_data") / "deflect_creativity"

EXPERIMENT_ID_BY_PROVIDER = {
    "openai": "openai_followup_simplestrat5_g2css3",
    "gemini": "gemini_followup_simplestrat5_g2css3",
    "anthropic": "anthropic_followup_simplestrat5_g2css3",
}

PROVIDER_ORDER = ["openai", "anthropic", "gemini"]

PROVIDER_LABEL = {
    "openai": "GPT-5.4",
    "anthropic": "Claude Sonnet 4.6",
    "gemini": "Gemini 2.5 Pro",
}

TASK_LABEL = {
    "slogan_smartphone": "Smartphone slogan",
    "slogan_soda": "Soda slogan",
    "slogan_blood_donation": "Blood donation slogan",
    "aut_shoe": "AUT: shoe",
    "aut_button": "AUT: button",
    "aut_key": "AUT: key",
    "aut_wooden_pencil": "AUT: wooden pencil",
    "aut_automobile_tire": "AUT: automobile tire",
    "story_jungle": "Story: jungle",
    "story_parachute": "Story: parachute",
    "story_horror": "Story: horror",
    "story_life_last_seconds": "Story: life / last seconds",
}

TASK_FAMILY = {
    "slogan_smartphone": "slogan",
    "slogan_soda": "slogan",
    "slogan_blood_donation": "slogan",
    "aut_shoe": "aut",
    "aut_button": "aut",
    "aut_key": "aut",
    "aut_wooden_pencil": "aut",
    "aut_automobile_tire": "aut",
    "story_jungle": "story",
    "story_parachute": "story",
    "story_horror": "story",
    "story_life_last_seconds": "story",
}

TASK_FAMILY_LABEL = {
    "aut": "Alternative uses",
    "story": "Story",
    "slogan": "Slogan",
}

AUT_ITEM_FOR_CAP = {
    "aut_shoe": "shoe",
    "aut_button": "button",
    "aut_key": "key",
    "aut_wooden_pencil": "wooden pencil",
    "aut_automobile_tire": "automobile tire",
}

AUT_COMMON_USE_FOR_TRACKING = {
    "aut_shoe": "used as footwear",
    "aut_button": "used to fasten things",
    "aut_key": "used to open a lock",
    "aut_wooden_pencil": "used for writing",
    "aut_automobile_tire": "used on the wheel of an automobile",
}

STORY_ITEM_FOR_CAP = {
    "story_jungle": "Write a story about an adventure in the jungle.",
    "story_parachute": "The parachute isn’t opening up.",
    "story_horror": "Write a short horror story designed to chill the bones.",
    "story_life_last_seconds": (
        "The first sentence must describe 100 years of a character's life. "
        "The next 7 sentences must describe the last 10 seconds of that character's life."
    ),
}

STORY_ITEM_ID_FOR_TRACKING = {
    "story_jungle": "adventure_jungle",
    "story_parachute": "parachute_not_opening",
    "story_horror": "horror_chill_bones",
    "story_life_last_seconds": "life_last_seconds",
}

QUALITY_RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

QUALITY_OUTPUT_ROOT = (
    Path("analysis_outputs")
    / "deflect_creativity_followup_quality"
    / f"quality_{QUALITY_RUN_ID}"
)

UPLOAD_DIR = QUALITY_OUTPUT_ROOT / "cap_upload_csvs"
AUT_UPLOAD_DIR = UPLOAD_DIR / "CLAUS_AUT_one_file_per_model"
STORY_UPLOAD_DIR = UPLOAD_DIR / "MAoSS_story_3parts_per_model"
TABLE_DIR = QUALITY_OUTPUT_ROOT / "tables"
MANIFEST_DIR = QUALITY_OUTPUT_ROOT / "manifests"

for d in [UPLOAD_DIR, AUT_UPLOAD_DIR, STORY_UPLOAD_DIR, TABLE_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Optional manual override if scanning misses a provider.
# Example:
# MANUAL_OUTPUT_FILES = {
#     "openai": Path(".../round2_new_baselines_outputs__....pkl"),
# }
MANUAL_OUTPUT_FILES = {}

STORY_N_PARTS_PER_MODEL = 3

print("Quality output root:")
print(QUALITY_OUTPUT_ROOT)

Quality output root:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333


In [3]:
def safe_slug(text: Any) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: Any, n: int = 24) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def word_count(text: Any) -> int:
    if pd.isna(text):
        return 0
    return len(re.findall(r"\b[\w'-]+\b", str(text)))


def rough_sentence_count(text: Any) -> int:
    if pd.isna(text):
        return 0
    parts = re.split(r"(?<=[.!?])\s+", str(text).strip())
    return len([p for p in parts if p.strip()])


def clean_cap_response_text(text: Any) -> str:
    """
    Conservative CAP upload cleaning:
    - preserve substantive wording
    - normalize Unicode
    - remove code fences
    - remove surrounding quote marks only if the entire response is quoted
    - collapse newlines/tabs into spaces
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFC", text)

    text = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)

    text = text.strip()

    if len(text) >= 2 and (
        (text[0] == text[-1] == '"')
        or (text[0] == text[-1] == "'")
        or (text[0] == "“" and text[-1] == "”")
        or (text[0] == "‘" and text[-1] == "’")
    ):
        text = text[1:-1].strip()

    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def write_cap_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path,
        index=False,
        encoding="utf-8",
        quoting=csv.QUOTE_MINIMAL,
        lineterminator="\n",
    )
    print(f"Wrote: {path}")
    print(f"Rows:  {len(df):,}")


def read_latest_output_file(path: Path) -> pd.DataFrame:
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported output file type: {path}")


def find_latest_followup_output(provider: str) -> Optional[Path]:
    if provider in MANUAL_OUTPUT_FILES:
        p = Path(MANUAL_OUTPUT_FILES[provider])
        if not p.exists():
            raise FileNotFoundError(f"Manual output file does not exist for {provider}: {p}")
        return p

    exp_id = EXPERIMENT_ID_BY_PROVIDER[provider]

    provider_root = FOLLOWUP_DATA_ROOT / provider
    if not provider_root.exists():
        return None

    patterns = [
        f"model_*/{exp_id}/run_*/05_compiled/*round2*new*baseline*outputs*.pkl",
        f"model_*/{exp_id}/run_*/05_compiled/*round2*new*baseline*outputs*.csv",
        f"model_*/{exp_id}/run_*/04_round2_new_baselines/parsed/*round2_new_baselines*parsed.pkl",
        f"model_*/{exp_id}/run_*/04_round2_new_baselines/parsed/*round2_new_baselines*parsed.csv",
    ]

    matches = []
    for pat in patterns:
        matches.extend(provider_root.glob(pat))

    matches = [p for p in matches if p.exists()]

    if not matches:
        return None

    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0]


def json_safe(obj: Any) -> Any:
    if obj is None:
        return None
    if isinstance(obj, (str, int, bool)):
        return obj
    if isinstance(obj, float):
        return None if (math.isnan(obj) or math.isinf(obj)) else obj
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        v = float(obj)
        return None if (math.isnan(v) or math.isinf(v)) else v
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return str(obj)

In [4]:
loaded_parts = []
source_rows = []

for provider in PROVIDER_ORDER:
    output_path = find_latest_followup_output(provider)

    if output_path is None:
        print(f"Skipping {provider}: no completed follow-up R2 output found yet.")
        continue

    df = read_latest_output_file(output_path).copy()

    df["provider"] = df.get("provider", provider)
    df["provider"] = df["provider"].fillna(provider).astype(str)

    if "model" not in df.columns:
        # Fallback from path if needed.
        model_match = re.search(r"model_([^/]+)", str(output_path))
        df["model"] = model_match.group(1) if model_match else "unknown_model"

    df["source_output_path"] = str(output_path)
    df["source_output_mtime"] = output_path.stat().st_mtime

    loaded_parts.append(df)

    source_rows.append({
        "provider": provider,
        "output_path": str(output_path),
        "n_rows": len(df),
        "modified_at": dt.datetime.fromtimestamp(output_path.stat().st_mtime).isoformat(),
    })

source_df = pd.DataFrame(source_rows)

display(source_df)

if not loaded_parts:
    raise RuntimeError("No completed follow-up outputs found. Check paths or MANUAL_OUTPUT_FILES.")

raw_followup = pd.concat(loaded_parts, ignore_index=True, sort=False)

print("Loaded follow-up rows:", raw_followup.shape)
display(raw_followup.head())
display(raw_followup.groupby(["provider", "model"], dropna=False).size().reset_index(name="n"))

,provider,output_path,n_rows,modified_at
0,openai,ai_data/deflect_creativity/openai/model_gpt-5....,7200,2026-05-23T13:41:05.168030
1,anthropic,ai_data/deflect_creativity/anthropic/model_cla...,7200,2026-05-23T14:10:03.777913
2,gemini,ai_data/deflect_creativity/gemini/model_gemini...,7200,2026-05-23T14:06:30.542257


Loaded follow-up rows: (21600, 55)


,provider,model,experiment_id,round,task_id,task_family,task_label,strategy,method,method_label,...,thinking_budget,include_thinking_config_in_batch,finish_reason,usage_prompt_token_count,usage_candidates_token_count,usage_thoughts_token_count,usage_cached_content_token_count,usage_total_token_count,batch_name,raw_record
0,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,2,slogan_smartphone,slogan,Smartphone slogan,vanilla,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,2,slogan_smartphone,slogan,Smartphone slogan,vanilla,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,2,slogan_smartphone,slogan,Smartphone slogan,vanilla,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,2,slogan_smartphone,slogan,Smartphone slogan,vanilla,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,openai,gpt-5.4,openai_followup_simplestrat5_g2css3,2,slogan_smartphone,slogan,Smartphone slogan,vanilla,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,provider,model,n
0,anthropic,claude-sonnet-4-6,7200
1,gemini,gemini-2.5-pro,7200
2,openai,gpt-5.4,7200


In [5]:
ideas = raw_followup.copy()

# Basic canonical fields.
ideas["provider"] = ideas["provider"].astype(str)
ideas["model"] = ideas["model"].astype(str)

if "experiment_id" not in ideas.columns:
    ideas["experiment_id"] = ideas["provider"].map(EXPERIMENT_ID_BY_PROVIDER)

if "round" not in ideas.columns:
    ideas["round"] = 2
else:
    ideas["round"] = ideas["round"].fillna(2).astype(int)

ideas["task_id"] = ideas["task_id"].astype(str)

if "task_family" not in ideas.columns:
    ideas["task_family"] = ideas["task_id"].map(TASK_FAMILY)
else:
    ideas["task_family"] = ideas["task_family"].fillna(ideas["task_id"].map(TASK_FAMILY)).astype(str)

if "task_label" not in ideas.columns:
    ideas["task_label"] = ideas["task_id"].map(TASK_LABEL)
else:
    ideas["task_label"] = ideas["task_label"].fillna(ideas["task_id"].map(TASK_LABEL))

ideas["task_family_label"] = ideas["task_family"].map(TASK_FAMILY_LABEL)
ideas["provider_label"] = ideas["provider"].map(PROVIDER_LABEL).fillna(ideas["provider"])

# Follow-up design fields.
if "method" not in ideas.columns:
    # Parsed files should have condition=method; use that if needed.
    ideas["method"] = ideas.get("condition", "unknown_method")

if "method_label" not in ideas.columns:
    ideas["method_label"] = ideas["method"]

if "condition" not in ideas.columns:
    ideas["condition"] = ideas["method"]

if "strategy" not in ideas.columns:
    raise RuntimeError("Missing strategy column.")

if "slot_id" not in ideas.columns:
    ideas["slot_id"] = ideas.groupby(["provider", "model", "task_id", "method", "strategy"]).cumcount().add(1).map(
        lambda x: f"slot_{x:03d}"
    )

if "slot_num" not in ideas.columns:
    ideas["slot_num"] = (
        ideas["slot_id"].astype(str).str.extract(r"(\d+)")[0]
        .astype(float)
        .fillna(ideas.groupby(["provider", "model", "task_id", "method", "strategy"]).cumcount().add(1))
        .astype(int)
    )

if "request_key" not in ideas.columns:
    ideas["request_key"] = ideas[
        ["provider", "model", "task_id", "method", "strategy", "slot_id"]
    ].astype(str).agg("|".join, axis=1).map(lambda x: "synthetic__" + stable_hash(x, 24))

# Text/status.
if "text" not in ideas.columns:
    raise RuntimeError("Missing text column.")

ideas["text_clean"] = ideas["text"].map(clean_cap_response_text)

if "status" not in ideas.columns:
    ideas["status"] = np.where(ideas["text_clean"].str.len().gt(0), "success", "empty_text")

ideas["is_success"] = ideas["status"].astype(str).eq("success") & ideas["text_clean"].str.len().gt(0)

# Stable quality ID. Include request_key so IDs remain stable across files.
quality_id_cols = [
    "provider", "model", "experiment_id", "task_id", "task_family",
    "method", "strategy", "slot_id", "request_key"
]
ideas["quality_row_id"] = ideas[quality_id_cols].astype(str).agg("|".join, axis=1).map(
    lambda x: stable_hash(x, 24)
)

ideas["response"] = ideas["text_clean"]
ideas["response_n_chars"] = ideas["response"].str.len()
ideas["response_word_count"] = ideas["response"].map(word_count)
ideas["rough_sentence_count"] = ideas["response"].map(rough_sentence_count)

# Keep only successful AUT/story rows.
quality_base = ideas[
    ideas["is_success"]
    & ideas["task_family"].isin(["aut", "story"])
].copy()

quality_base = quality_base.sort_values(
    [
        "provider", "model", "task_family", "task_id",
        "method", "strategy", "slot_num", "request_key"
    ]
).reset_index(drop=True)

quality_base["cap_upload_row"] = np.arange(1, len(quality_base) + 1)

if quality_base["quality_row_id"].duplicated().any():
    dupes = quality_base[quality_base["quality_row_id"].duplicated(keep=False)]
    display(dupes[quality_id_cols + ["quality_row_id", "response"]].head(50))
    raise RuntimeError("Duplicate quality_row_id values found.")

print("Quality-base rows:", quality_base.shape)

display(
    quality_base
    .groupby(["provider", "model", "task_family", "task_id", "method", "strategy"], observed=True)
    .agg(
        n=("quality_row_id", "count"),
        min_chars=("response_n_chars", "min"),
        median_chars=("response_n_chars", "median"),
        max_chars=("response_n_chars", "max"),
        mean_words=("response_word_count", "mean"),
    )
    .reset_index()
)

Quality-base rows: (16200, 65)


,provider,model,task_family,task_id,method,strategy,n,min_chars,median_chars,max_chars,mean_words
0,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,g2_css_static3,diverge,150,108,164.5,258,27.120000
1,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,g2_css_static3,vanilla,150,80,150.0,239,24.820000
2,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,simplestrat5,diverge,150,91,195.5,288,29.040000
3,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,simplestrat5,vanilla,150,111,168.0,261,25.586667
4,anthropic,claude-sonnet-4-6,aut,aut_button,g2_css_static3,diverge,150,74,106.5,171,19.640000
...,...,...,...,...,...,...,...,...,...,...,...
103,openai,gpt-5.4,story,story_life_last_seconds,simplestrat5,vanilla,150,984,1375.5,1795,241.600000
104,openai,gpt-5.4,story,story_parachute,g2_css_static3,diverge,150,958,1230.5,1455,220.020000
105,openai,gpt-5.4,story,story_parachute,g2_css_static3,vanilla,150,864,1124.0,1378,204.046667
106,openai,gpt-5.4,story,story_parachute,simplestrat5,diverge,150,906,1288.5,1641,235.406667


In [6]:
# -------------------------------------------------------------------
# AUT / CLAUS
# -------------------------------------------------------------------

aut_df = quality_base[quality_base["task_family"].eq("aut")].copy()

aut_df["item"] = aut_df["task_id"].map(AUT_ITEM_FOR_CAP)
aut_df["common_use"] = aut_df["task_id"].map(AUT_COMMON_USE_FOR_TRACKING)

missing_aut_items = sorted(aut_df.loc[aut_df["item"].isna(), "task_id"].unique())
if missing_aut_items:
    raise RuntimeError(f"Missing AUT item mappings: {missing_aut_items}")

# -------------------------------------------------------------------
# Story / MAoSS
# -------------------------------------------------------------------

story_df = quality_base[quality_base["task_family"].eq("story")].copy()

story_df["item"] = story_df["task_id"].map(STORY_ITEM_FOR_CAP)
story_df["item_id"] = story_df["task_id"].map(STORY_ITEM_ID_FOR_TRACKING)

missing_story_items = sorted(story_df.loc[story_df["item"].isna(), "task_id"].unique())
if missing_story_items:
    raise RuntimeError(f"Missing story item mappings: {missing_story_items}")

# Tracking columns: keep item,response first for CAP; everything after is for merging/audit.
COMMON_TRACKING_COLS = [
    "quality_row_id",
    "cap_upload_row",
    "provider",
    "provider_label",
    "model",
    "experiment_id",
    "task_id",
    "task_label",
    "task_family",
    "task_family_label",
    "method",
    "method_label",
    "condition",
    "strategy",
    "round",
    "slot_id",
    "slot_num",
    "stratum_id",
    "anchor_count",
    "context_count",
    "request_key",
    "status",
    "finish_reason",
    "usage_prompt_token_count",
    "usage_candidates_token_count",
    "usage_thoughts_token_count",
    "usage_total_token_count",
    "response_word_count",
    "response_n_chars",
    "rough_sentence_count",
    "batch_id",
    "batch_name",
    "batch_custom_id",
    "batch_output_file",
    "source_output_path",
]

aut_extra_cols = ["common_use"]
story_extra_cols = ["item_id"]

aut_cols = ["item", "response"] + [
    c for c in COMMON_TRACKING_COLS + aut_extra_cols
    if c in aut_df.columns
]

story_cols = ["item", "response"] + [
    c for c in COMMON_TRACKING_COLS + story_extra_cols
    if c in story_df.columns
]

aut_cap_full = aut_df[aut_cols].copy()
story_cap_full = story_df[story_cols].copy()

print("AUT CAP rows:", aut_cap_full.shape)
display(aut_cap_full.head())
display(aut_cap_full.groupby(["provider", "model", "method", "strategy", "task_id"]).size().reset_index(name="n"))

print("Story CAP rows:", story_cap_full.shape)
display(story_cap_full.head())
display(story_cap_full.groupby(["provider", "model", "method", "strategy", "task_id"]).size().reset_index(name="n"))

AUT CAP rows: (9000, 38)


,item,response,quality_row_id,cap_upload_row,provider,provider_label,model,experiment_id,task_id,task_label,...,usage_total_token_count,response_word_count,response_n_chars,rough_sentence_count,batch_id,batch_name,batch_custom_id,batch_output_file,source_output_path,common_use
0,automobile tire,Submerged in a lake or ocean to serve as an ar...,4c5c47b462c5dbcb61e85596,1,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,NaN,26,157,1,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__7c4900a15450a4ff3beaef61,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,used on the wheel of an automobile
1,automobile tire,Suspend a tire horizontally from a ceiling bea...,3a1a65dd7306d66bcdf9c52e,2,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,NaN,42,238,1,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__a8f6c634ee14384215ab0f42,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,used on the wheel of an automobile
2,automobile tire,Submerged in a lake or ocean to serve as an ar...,4efb180ee106d337d440c076,3,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,NaN,23,137,1,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__bfffe375a75cd7e8d2b80a83,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,used on the wheel of an automobile
3,automobile tire,Submerged in a lake or ocean as an artificial ...,44d9b3bec75d6e38a2ce781e,4,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,NaN,27,163,1,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__912738d2ad15f27502c5f7c8,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,used on the wheel of an automobile
4,automobile tire,Suspend a tire horizontally from a ceiling bea...,5f90b0cadfc4501af508198b,5,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,NaN,26,152,1,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__e78f5f52b1831a61b27fa3d9,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,used on the wheel of an automobile


,provider,model,method,strategy,task_id,n
0,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,aut_automobile_tire,150
1,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,aut_button,150
2,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,aut_key,150
3,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,aut_shoe,150
4,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,aut_wooden_pencil,150
5,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,aut_automobile_tire,150
6,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,aut_button,150
7,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,aut_key,150
8,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,aut_shoe,150
9,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,aut_wooden_pencil,150


Story CAP rows: (7200, 38)


,item,response,quality_row_id,cap_upload_row,provider,provider_label,model,experiment_id,task_id,task_label,...,usage_total_token_count,response_word_count,response_n_chars,rough_sentence_count,batch_id,batch_name,batch_custom_id,batch_output_file,source_output_path,item_id
3000,Write a short horror story designed to chill t...,"The voicemail was only four seconds long, and ...",1add549d4dd2c98cc5acc93b,3001,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,NaN,210,1133,6,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__31e2ca7723a4023ad1895a88,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,horror_chill_bones
3001,Write a short horror story designed to chill t...,"Every night for a month, I fell asleep to the ...",3bda51943a8da0c318cc07be,3002,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,NaN,213,1171,8,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__bec7d1033b4d0ea3ad881bad,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,horror_chill_bones
3002,Write a short horror story designed to chill t...,"The summer my grandmother died, she made me pr...",93233f653455cf6e9e3abda0,3003,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,NaN,242,1322,9,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__4573c43afad0734c5a5ef590,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,horror_chill_bones
3003,Write a short horror story designed to chill t...,"The voicemail was only twelve seconds long, le...",3ce88e6dcaf5e77c641b80c5,3004,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,NaN,204,1134,8,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__aca0a90a86f85023bcc7da07,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,horror_chill_bones
3004,Write a short horror story designed to chill t...,"The voicemail was only four seconds long, left...",75dd7079729536e13a2bf135,3005,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,NaN,216,1196,8,msgbatch_01Wfo2HGYALRrfMLqEukHD4Y,NaN,r2new__6fc522cb9f938d40179cee99,ai_data/deflect_creativity/anthropic/model_cla...,ai_data/deflect_creativity/anthropic/model_cla...,horror_chill_bones


,provider,model,method,strategy,task_id,n
0,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,story_horror,150
1,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,story_jungle,150
2,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,story_life_last_seconds,150
3,anthropic,claude-sonnet-4-6,g2_css_static3,diverge,story_parachute,150
4,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,story_horror,150
5,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,story_jungle,150
6,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,story_life_last_seconds,150
7,anthropic,claude-sonnet-4-6,g2_css_static3,vanilla,story_parachute,150
8,anthropic,claude-sonnet-4-6,simplestrat5,diverge,story_horror,150
9,anthropic,claude-sonnet-4-6,simplestrat5,diverge,story_jungle,150


In [7]:
def cap_upload_qc(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    required = ["item", "response"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"{dataset_name} missing required columns: {missing}")

    blank_item = df["item"].isna() | df["item"].astype(str).str.strip().eq("")
    blank_response = df["response"].isna() | df["response"].astype(str).str.strip().eq("")

    qc = pd.DataFrame([{
        "dataset": dataset_name,
        "n_rows": len(df),
        "n_blank_item": int(blank_item.sum()),
        "n_blank_response": int(blank_response.sum()),
        "n_unique_items": int(df["item"].nunique(dropna=True)),
        "min_response_chars": int(df["response"].astype(str).str.len().min()) if len(df) else 0,
        "median_response_chars": float(df["response"].astype(str).str.len().median()) if len(df) else 0,
        "max_response_chars": int(df["response"].astype(str).str.len().max()) if len(df) else 0,
    }])

    if blank_item.any() or blank_response.any():
        display(df.loc[blank_item | blank_response, required + [c for c in ["provider", "model", "task_id"] if c in df.columns]].head(50))
        raise RuntimeError(f"{dataset_name} contains blank item or response values.")

    return qc


qc_tables = []

if not aut_cap_full.empty:
    qc_tables.append(cap_upload_qc(aut_cap_full, "CLAUS_AUT_followup_full"))

if not story_cap_full.empty:
    qc_tables.append(cap_upload_qc(story_cap_full, "MAoSS_story_followup_full"))

upload_qc = pd.concat(qc_tables, ignore_index=True) if qc_tables else pd.DataFrame()

qc_path = TABLE_DIR / "cap_upload_qc_followup.csv"
upload_qc.to_csv(qc_path, index=False)

display(upload_qc)
print("Saved QC:", qc_path)

cell_counts = (
    pd.concat(
        [
            aut_cap_full.assign(cap_tool="CLAUS") if not aut_cap_full.empty else pd.DataFrame(),
            story_cap_full.assign(cap_tool="MAoSS") if not story_cap_full.empty else pd.DataFrame(),
        ],
        ignore_index=True,
        sort=False,
    )
    .groupby(["cap_tool", "provider", "model", "task_family", "task_id", "method", "strategy"], observed=True)
    .agg(n=("quality_row_id", "count"))
    .reset_index()
)

counts_path = TABLE_DIR / "cap_upload_counts_by_cell_followup.csv"
cell_counts.to_csv(counts_path, index=False)

display(cell_counts)
print("Saved counts:", counts_path)

,dataset,n_rows,n_blank_item,n_blank_response,n_unique_items,min_response_chars,median_response_chars,max_response_chars
0,CLAUS_AUT_followup_full,9000,0,0,5,33,102.0,290
1,MAoSS_story_followup_full,7200,0,0,4,481,1207.0,2199


Saved QC: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/tables/cap_upload_qc_followup.csv


,cap_tool,provider,model,task_family,task_id,method,strategy,n
0,CLAUS,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,g2_css_static3,diverge,150
1,CLAUS,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,g2_css_static3,vanilla,150
2,CLAUS,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,simplestrat5,diverge,150
3,CLAUS,anthropic,claude-sonnet-4-6,aut,aut_automobile_tire,simplestrat5,vanilla,150
4,CLAUS,anthropic,claude-sonnet-4-6,aut,aut_button,g2_css_static3,diverge,150
...,...,...,...,...,...,...,...,...
103,MAoSS,openai,gpt-5.4,story,story_life_last_seconds,simplestrat5,vanilla,150
104,MAoSS,openai,gpt-5.4,story,story_parachute,g2_css_static3,diverge,150
105,MAoSS,openai,gpt-5.4,story,story_parachute,g2_css_static3,vanilla,150
106,MAoSS,openai,gpt-5.4,story,story_parachute,simplestrat5,diverge,150


Saved counts: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/tables/cap_upload_counts_by_cell_followup.csv


In [8]:
# -------------------------------------------------------------------
# AUT / CLAUS: one CSV per provider-model
# -------------------------------------------------------------------

aut_written = []

if aut_cap_full.empty:
    print("No AUT rows found.")
else:
    for (provider, model), model_df in aut_cap_full.groupby(["provider", "model"], observed=True):
        model_df = model_df.sort_values(
            ["task_id", "method", "strategy", "slot_num", "request_key"]
        ).reset_index(drop=True)

        model_slug = safe_slug(f"{provider}_{model}")

        out_path = AUT_UPLOAD_DIR / f"CLAUS_upload_AUT_{model_slug}_FULL_with_ids.csv"
        write_cap_csv(model_df, out_path)

        # Optional fallback key file, in case a website strips extra columns.
        key_path = AUT_UPLOAD_DIR / f"CLAUS_upload_AUT_{model_slug}_ROW_KEY.csv"
        model_df.drop(columns=["item", "response"]).to_csv(key_path, index=False)

        aut_written.append({
            "provider": provider,
            "model": model,
            "task_family": "aut",
            "cap_tool": "CLAUS",
            "file_type": "full_with_ids",
            "path": str(out_path),
            "key_path": str(key_path),
            "n_rows": len(model_df),
            "n_items": int(model_df["item"].nunique()),
            "n_tasks": int(model_df["task_id"].nunique()),
        })

aut_written_df = pd.DataFrame(aut_written)
aut_manifest_path = MANIFEST_DIR / "CLAUS_AUT_files_written_one_per_model.csv"
aut_written_df.to_csv(aut_manifest_path, index=False)

display(aut_written_df)
print("Saved AUT manifest:", aut_manifest_path)

Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/CLAUS_AUT_one_file_per_model/CLAUS_upload_AUT_anthropic_claude_sonnet_4_6_FULL_with_ids.csv
Rows:  3,000
Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/CLAUS_AUT_one_file_per_model/CLAUS_upload_AUT_gemini_gemini_2_5_pro_FULL_with_ids.csv
Rows:  3,000
Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/CLAUS_AUT_one_file_per_model/CLAUS_upload_AUT_openai_gpt_5_4_FULL_with_ids.csv
Rows:  3,000


,provider,model,task_family,cap_tool,file_type,path,key_path,n_rows,n_items,n_tasks
0,anthropic,claude-sonnet-4-6,aut,CLAUS,full_with_ids,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,3000,5,5
1,gemini,gemini-2.5-pro,aut,CLAUS,full_with_ids,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,3000,5,5
2,openai,gpt-5.4,aut,CLAUS,full_with_ids,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,3000,5,5


Saved AUT manifest: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/manifests/CLAUS_AUT_files_written_one_per_model.csv


In [9]:
# -------------------------------------------------------------------
# Story / MAoSS: three balanced parts per provider-model
# -------------------------------------------------------------------

story_written = []

if story_cap_full.empty:
    print("No story rows found.")
else:
    for (provider, model), model_df in story_cap_full.groupby(["provider", "model"], observed=True):
        # Sort by cell first, then assign round-robin split. This makes each split
        # contain a balanced mixture of tasks, methods, and strategies.
        model_df = model_df.sort_values(
            ["task_id", "method", "strategy", "slot_num", "request_key"]
        ).reset_index(drop=True)

        model_df["maoss_provider_upload_row"] = np.arange(1, len(model_df) + 1)
        model_df["maoss_split_index"] = (np.arange(len(model_df)) % STORY_N_PARTS_PER_MODEL) + 1
        model_df["maoss_split_total"] = STORY_N_PARTS_PER_MODEL

        model_slug = safe_slug(f"{provider}_{model}")

        for part_idx in range(1, STORY_N_PARTS_PER_MODEL + 1):
            part_df = model_df[model_df["maoss_split_index"].eq(part_idx)].copy()
            part_df = part_df.sort_values(
                ["task_id", "method", "strategy", "slot_num", "request_key"]
            ).reset_index(drop=True)
            part_df["maoss_split_row"] = np.arange(1, len(part_df) + 1)

            out_path = (
                STORY_UPLOAD_DIR
                / f"MAoSS_upload_story_{model_slug}_part{part_idx:02d}of{STORY_N_PARTS_PER_MODEL:02d}_FULL_with_ids.csv"
            )
            write_cap_csv(part_df, out_path)

            key_path = (
                STORY_UPLOAD_DIR
                / f"MAoSS_upload_story_{model_slug}_part{part_idx:02d}of{STORY_N_PARTS_PER_MODEL:02d}_ROW_KEY.csv"
            )
            part_df.drop(columns=["item", "response"]).to_csv(key_path, index=False)

            story_written.append({
                "provider": provider,
                "model": model,
                "task_family": "story",
                "cap_tool": "MAoSS",
                "file_type": "full_with_ids",
                "split_index": part_idx,
                "split_total": STORY_N_PARTS_PER_MODEL,
                "path": str(out_path),
                "key_path": str(key_path),
                "n_rows": len(part_df),
                "n_items": int(part_df["item"].nunique()),
                "n_tasks": int(part_df["task_id"].nunique()),
            })

story_written_df = pd.DataFrame(story_written)
story_manifest_path = MANIFEST_DIR / "MAoSS_story_files_written_3parts_per_model.csv"
story_written_df.to_csv(story_manifest_path, index=False)

display(story_written_df)
print("Saved story manifest:", story_manifest_path)

Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/MAoSS_story_3parts_per_model/MAoSS_upload_story_anthropic_claude_sonnet_4_6_part01of03_FULL_with_ids.csv
Rows:  800
Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/MAoSS_story_3parts_per_model/MAoSS_upload_story_anthropic_claude_sonnet_4_6_part02of03_FULL_with_ids.csv
Rows:  800
Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/MAoSS_story_3parts_per_model/MAoSS_upload_story_anthropic_claude_sonnet_4_6_part03of03_FULL_with_ids.csv
Rows:  800
Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/MAoSS_story_3parts_per_model/MAoSS_upload_story_gemini_gemini_2_5_pro_part01of03_FULL_with_ids.csv
Rows:  800
Wrote: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_upload_csvs/MAoSS_story_3parts_per_model/MAoSS_upload_

,provider,model,task_family,cap_tool,file_type,split_index,split_total,path,key_path,n_rows,n_items,n_tasks
0,anthropic,claude-sonnet-4-6,story,MAoSS,full_with_ids,1,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
1,anthropic,claude-sonnet-4-6,story,MAoSS,full_with_ids,2,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
2,anthropic,claude-sonnet-4-6,story,MAoSS,full_with_ids,3,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
3,gemini,gemini-2.5-pro,story,MAoSS,full_with_ids,1,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
4,gemini,gemini-2.5-pro,story,MAoSS,full_with_ids,2,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
5,gemini,gemini-2.5-pro,story,MAoSS,full_with_ids,3,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
6,openai,gpt-5.4,story,MAoSS,full_with_ids,1,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
7,openai,gpt-5.4,story,MAoSS,full_with_ids,2,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
8,openai,gpt-5.4,story,MAoSS,full_with_ids,3,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4


Saved story manifest: analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/manifests/MAoSS_story_files_written_3parts_per_model.csv


In [10]:
final_manifest = {
    "quality_run_id": QUALITY_RUN_ID,
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "quality_output_root": str(QUALITY_OUTPUT_ROOT),
    "upload_dir": str(UPLOAD_DIR),
    "aut_upload_dir": str(AUT_UPLOAD_DIR),
    "story_upload_dir": str(STORY_UPLOAD_DIR),
    "followup_data_root": str(FOLLOWUP_DATA_ROOT),
    "experiment_id_by_provider": EXPERIMENT_ID_BY_PROVIDER,
    "source_outputs": source_rows,
    "included_providers": sorted(quality_base["provider"].unique().tolist()),
    "missing_providers": [
        p for p in PROVIDER_ORDER
        if p not in set(quality_base["provider"].unique())
    ],
    "aut_manifest_path": str(aut_manifest_path),
    "story_manifest_path": str(story_manifest_path),
    "qc_path": str(qc_path),
    "counts_path": str(counts_path),
    "item_mappings": {
        "aut_item_for_cap": AUT_ITEM_FOR_CAP,
        "aut_common_use_for_tracking": AUT_COMMON_USE_FOR_TRACKING,
        "story_item_for_cap": STORY_ITEM_FOR_CAP,
        "story_item_id_for_tracking": STORY_ITEM_ID_FOR_TRACKING,
    },
    "notes": [
        "AUT files are written one CSV per provider-model.",
        "Story files are written as three balanced CSV parts per provider-model.",
        "Each upload CSV begins with CAP-required columns item,response.",
        "Tracking columns are preserved after item,response for merging returned quality scores.",
        "If CAP strips extra columns, use the matching ROW_KEY file and row order to merge.",
        "Rerun this notebook after Anthropic R2 finishes; it will be picked up automatically.",
    ],
}

final_manifest_path = MANIFEST_DIR / "quality_upload_manifest_followup_aut_story.json"

with open(final_manifest_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(final_manifest), f, indent=2, ensure_ascii=False)

print("Saved final manifest:")
print(final_manifest_path)

print("\nAUT files:")
display(aut_written_df)

print("\nStory files:")
display(story_written_df)

print("\nMissing providers, expected while Anthropic is still pending:")
print(final_manifest["missing_providers"])

Saved final manifest:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/manifests/quality_upload_manifest_followup_aut_story.json

AUT files:


,provider,model,task_family,cap_tool,file_type,path,key_path,n_rows,n_items,n_tasks
0,anthropic,claude-sonnet-4-6,aut,CLAUS,full_with_ids,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,3000,5,5
1,gemini,gemini-2.5-pro,aut,CLAUS,full_with_ids,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,3000,5,5
2,openai,gpt-5.4,aut,CLAUS,full_with_ids,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,3000,5,5



Story files:


,provider,model,task_family,cap_tool,file_type,split_index,split_total,path,key_path,n_rows,n_items,n_tasks
0,anthropic,claude-sonnet-4-6,story,MAoSS,full_with_ids,1,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
1,anthropic,claude-sonnet-4-6,story,MAoSS,full_with_ids,2,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
2,anthropic,claude-sonnet-4-6,story,MAoSS,full_with_ids,3,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
3,gemini,gemini-2.5-pro,story,MAoSS,full_with_ids,1,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
4,gemini,gemini-2.5-pro,story,MAoSS,full_with_ids,2,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
5,gemini,gemini-2.5-pro,story,MAoSS,full_with_ids,3,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
6,openai,gpt-5.4,story,MAoSS,full_with_ids,1,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
7,openai,gpt-5.4,story,MAoSS,full_with_ids,2,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4
8,openai,gpt-5.4,story,MAoSS,full_with_ids,3,3,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,800,4,4



Missing providers, expected while Anthropic is still pending:
[]


In [11]:
# -------------------------------------------------------------------
# Returned CAP files
# -------------------------------------------------------------------

RETURNED_DIR = QUALITY_OUTPUT_ROOT / "cap_returned_csvs"
RETURNED_AUT_DIR = RETURNED_DIR / "CLAUS_AUT"
RETURNED_STORY_DIR = RETURNED_DIR / "MAoSS_story"

ANALYSIS_READY_DIR = QUALITY_OUTPUT_ROOT / "analysis_ready_quality_scores"
ANALYSIS_READY_AUT_DIR = ANALYSIS_READY_DIR / "AUT_CLAUS"
ANALYSIS_READY_STORY_DIR = ANALYSIS_READY_DIR / "story_MAoSS"

for d in [RETURNED_DIR, RETURNED_AUT_DIR, RETURNED_STORY_DIR, ANALYSIS_READY_DIR, ANALYSIS_READY_AUT_DIR, ANALYSIS_READY_STORY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Put returned AUT files here:")
print(RETURNED_AUT_DIR)

print("\nPut returned story files here:")
print(RETURNED_STORY_DIR)

print("\nAnalysis-ready outputs will be written here:")
print(ANALYSIS_READY_DIR)

Put returned AUT files here:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_returned_csvs/CLAUS_AUT

Put returned story files here:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/cap_returned_csvs/MAoSS_story

Analysis-ready outputs will be written here:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores


In [12]:
def read_returned_csv(path: Path) -> pd.DataFrame:
    """
    Robust returned CSV reader. CAP exports are usually UTF-8, but this gives
    a fallback for odd encodings.
    """
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1")


def normalize_returned_filename(path: Path) -> str:
    """
    Map returned filename back to original upload filename.
    Preferred format:
        RETURNED__<original_upload_filename>
    """
    name = path.name

    prefixes = [
        "RETURNED__",
        "returned__",
        "Returned__",
        "RETURNED_",
        "returned_",
    ]

    for p in prefixes:
        if name.startswith(p):
            return name[len(p):]

    return name


def find_matching_upload_file(returned_path: Path, upload_dir: Path) -> Path:
    original_name = normalize_returned_filename(returned_path)
    candidate = upload_dir / original_name

    if candidate.exists():
        return candidate

    # Fallback: remove common browser suffixes like " (1)"
    simplified = re.sub(r"\s+\(\d+\)(?=\.csv$)", "", original_name)
    candidate = upload_dir / simplified

    if candidate.exists():
        return candidate

    # Fallback: fuzzy-ish stem containment
    returned_stem = Path(original_name).stem
    upload_candidates = sorted(upload_dir.glob("*.csv"))

    upload_candidates = [
        p for p in upload_candidates
        if "ROW_KEY" not in p.name
        and (p.stem in returned_stem or returned_stem in p.stem)
    ]

    if len(upload_candidates) == 1:
        return upload_candidates[0]

    raise FileNotFoundError(
        f"Could not find matching upload file for returned file:\n"
        f"Returned: {returned_path}\n"
        f"Expected original name: {original_name}\n"
        f"Upload dir: {upload_dir}"
    )


def find_matching_key_file(upload_path: Path) -> Optional[Path]:
    key_name = upload_path.name.replace("_FULL_with_ids.csv", "_ROW_KEY.csv")
    key_path = upload_path.parent / key_name
    return key_path if key_path.exists() else None


def standardize_colnames(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep original columns, but strip whitespace from column names.
    """
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def add_row_order(df: pd.DataFrame, colname: str = "cap_returned_row") -> pd.DataFrame:
    out = df.copy()
    out[colname] = np.arange(1, len(out) + 1)
    return out


def prefix_returned_columns(
    returned_df: pd.DataFrame,
    upload_df: pd.DataFrame,
    keep_unprefixed: list[str] = ["quality_row_id", "item", "response"],
) -> pd.DataFrame:
    """
    Prefix returned-only columns with cap_returned__.
    If a returned column duplicates an upload/tracking column, preserve the returned
    value but prefix it unless it is one of the key columns.
    """
    returned_df = returned_df.copy()

    rename = {}
    upload_cols = set(upload_df.columns)

    for c in returned_df.columns:
        if c in keep_unprefixed:
            continue
        if c in upload_cols or c in {"cap_returned_row"}:
            rename[c] = f"cap_returned__{c}"
        else:
            rename[c] = f"cap_returned__{c}"

    return returned_df.rename(columns=rename)


def merge_returned_with_upload(returned_path: Path, upload_dir: Path) -> pd.DataFrame:
    """
    Merge a returned CAP file to the original upload file.

    Merge priority:
    1. quality_row_id if preserved in returned file.
    2. row order using the original upload file.
    3. row order using ROW_KEY file, if needed.
    """
    upload_path = find_matching_upload_file(returned_path, upload_dir)
    key_path = find_matching_key_file(upload_path)

    returned = standardize_colnames(read_returned_csv(returned_path))
    upload = standardize_colnames(pd.read_csv(upload_path))

    returned = add_row_order(returned, "cap_returned_row")
    upload = add_row_order(upload, "cap_upload_file_row")

    if len(returned) != len(upload):
        raise RuntimeError(
            f"Returned/upload row mismatch:\n"
            f"Returned: {returned_path} ({len(returned)} rows)\n"
            f"Upload:   {upload_path} ({len(upload)} rows)"
        )

    if "quality_row_id" in returned.columns and "quality_row_id" in upload.columns:
        if returned["quality_row_id"].duplicated().any():
            raise RuntimeError(f"Duplicate quality_row_id in returned file: {returned_path}")
        if upload["quality_row_id"].duplicated().any():
            raise RuntimeError(f"Duplicate quality_row_id in upload file: {upload_path}")

        returned_prefixed = prefix_returned_columns(returned, upload)

        merged = upload.merge(
            returned_prefixed,
            on="quality_row_id",
            how="left",
            validate="one_to_one",
            suffixes=("", "__upload_duplicate"),
        )

        merged["merge_method"] = "quality_row_id"

    else:
        returned_prefixed = prefix_returned_columns(returned, upload)

        # If CAP stripped all tracking columns, row order is the intended fallback.
        upload_ordered = upload.reset_index(drop=True)
        returned_ordered = returned_prefixed.reset_index(drop=True)

        # Drop duplicate unprefixed item/response from returned side if present.
        for c in ["item", "response"]:
            if c in returned_ordered.columns:
                returned_ordered = returned_ordered.drop(columns=[c])

        merged = pd.concat([upload_ordered, returned_ordered], axis=1)
        merged["merge_method"] = "row_order"

        if key_path is not None:
            key_df = add_row_order(standardize_colnames(pd.read_csv(key_path)), "cap_key_file_row")
            if len(key_df) == len(merged):
                merged["key_file_path"] = str(key_path)

    merged["returned_file_path"] = str(returned_path)
    merged["upload_file_path"] = str(upload_path)
    merged["upload_file_name"] = upload_path.name
    merged["returned_file_name"] = returned_path.name

    return merged


def list_returned_files(returned_dir: Path) -> list[Path]:
    return sorted([
        p for p in returned_dir.glob("*.csv")
        if not p.name.startswith(".")
    ])


def candidate_score_columns(df: pd.DataFrame) -> list[str]:
    """
    Identify likely returned numeric score columns.
    We intentionally keep this broad because CAP tools may change column names.
    """
    exclude_exact = {
        "cap_upload_row",
        "cap_upload_file_row",
        "cap_returned_row",
        "slot_num",
        "round",
        "stratum_id",
        "anchor_count",
        "context_count",
        "response_word_count",
        "response_n_chars",
        "rough_sentence_count",
        "usage_prompt_token_count",
        "usage_candidates_token_count",
        "usage_thoughts_token_count",
        "usage_total_token_count",
        "usage_input_tokens",
        "usage_output_tokens",
    }

    likely_name_bits = [
        "score", "rating", "original", "creativ", "novel", "semantic",
        "quality", "appropr", "surpris", "utility", "interesting",
        "maoss", "claus", "distance"
    ]

    out = []
    for c in df.columns:
        c_str = str(c)
        c_low = c_str.lower()

        if c_str in exclude_exact:
            continue

        if not (
            c_str.startswith("cap_returned__")
            or any(bit in c_low for bit in likely_name_bits)
        ):
            continue

        numeric = pd.to_numeric(df[c], errors="coerce")
        n_numeric = numeric.notna().sum()

        if n_numeric > 0 and n_numeric / max(len(df), 1) >= 0.5:
            out.append(c)

    return out

In [13]:
aut_returned_files = list_returned_files(RETURNED_AUT_DIR)

print("Returned AUT files found:", len(aut_returned_files))
for p in aut_returned_files:
    print(" -", p.name)

if len(aut_returned_files) == 0:
    raise RuntimeError(f"No returned AUT files found in {RETURNED_AUT_DIR}")

aut_merged_parts = []

for returned_path in aut_returned_files:
    merged = merge_returned_with_upload(
        returned_path=returned_path,
        upload_dir=AUT_UPLOAD_DIR,
    )
    merged["cap_tool"] = "CLAUS"
    merged["quality_task_family"] = "aut"
    aut_merged_parts.append(merged)

aut_quality_rows = pd.concat(aut_merged_parts, ignore_index=True, sort=False)

if aut_quality_rows["quality_row_id"].duplicated().any():
    dupes = aut_quality_rows[aut_quality_rows["quality_row_id"].duplicated(keep=False)]
    display(dupes[["quality_row_id", "returned_file_name", "upload_file_name", "item", "response"]].head(50))
    raise RuntimeError("Duplicate AUT quality_row_id values after merging returned files.")

aut_score_cols = candidate_score_columns(aut_quality_rows)

print("Merged AUT rows:", len(aut_quality_rows))
print("Candidate AUT score columns:")
print(aut_score_cols)

display(aut_quality_rows.head())
display(
    aut_quality_rows
    .groupby(["provider", "model", "task_id", "method", "strategy"], observed=True)
    .size()
    .reset_index(name="n")
)

Returned AUT files found: 3
 - RETURNED__CLAUS_upload_AUT_anthropic_claude_sonnet_4_6_FULL_with_ids.csv
 - RETURNED__CLAUS_upload_AUT_gemini_gemini_2_5_pro_FULL_with_ids.csv
 - RETURNED__CLAUS_upload_AUT_openai_gpt_5_4_FULL_with_ids.csv
Merged AUT rows: 9000
Candidate AUT score columns:
['cap_returned__cap_upload_row', 'cap_returned__round', 'cap_returned__slot_num', 'cap_returned__stratum_id', 'cap_returned__anchor_count', 'cap_returned__context_count', 'cap_returned__response_word_count', 'cap_returned__response_n_chars', 'cap_returned__rough_sentence_count', 'cap_returned__prediction', 'cap_returned__cap_returned_row']


,item,response,quality_row_id,cap_upload_row,provider,provider_label,model,experiment_id,task_id,task_label,...,cap_returned__modelname,cap_returned__text,cap_returned__cap_returned_row,merge_method,returned_file_path,upload_file_path,upload_file_name,returned_file_name,cap_tool,quality_task_family
0,automobile tire,Submerged in a lake or ocean to serve as an ar...,4c5c47b462c5dbcb61e85596,1,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,CLAUS,"Identify how surprising, creative, unexpected,...",1,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,CLAUS_upload_AUT_anthropic_claude_sonnet_4_6_F...,RETURNED__CLAUS_upload_AUT_anthropic_claude_so...,CLAUS,aut
1,automobile tire,Suspend a tire horizontally from a ceiling bea...,3a1a65dd7306d66bcdf9c52e,2,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,CLAUS,"Identify how surprising, creative, unexpected,...",2,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,CLAUS_upload_AUT_anthropic_claude_sonnet_4_6_F...,RETURNED__CLAUS_upload_AUT_anthropic_claude_so...,CLAUS,aut
2,automobile tire,Submerged in a lake or ocean to serve as an ar...,4efb180ee106d337d440c076,3,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,CLAUS,"Identify how surprising, creative, unexpected,...",3,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,CLAUS_upload_AUT_anthropic_claude_sonnet_4_6_F...,RETURNED__CLAUS_upload_AUT_anthropic_claude_so...,CLAUS,aut
3,automobile tire,Submerged in a lake or ocean as an artificial ...,44d9b3bec75d6e38a2ce781e,4,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,CLAUS,"Identify how surprising, creative, unexpected,...",4,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,CLAUS_upload_AUT_anthropic_claude_sonnet_4_6_F...,RETURNED__CLAUS_upload_AUT_anthropic_claude_so...,CLAUS,aut
4,automobile tire,Suspend a tire horizontally from a ceiling bea...,5f90b0cadfc4501af508198b,5,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,aut_automobile_tire,AUT: automobile tire,...,CLAUS,"Identify how surprising, creative, unexpected,...",5,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,CLAUS_upload_AUT_anthropic_claude_sonnet_4_6_F...,RETURNED__CLAUS_upload_AUT_anthropic_claude_so...,CLAUS,aut


,provider,model,task_id,method,strategy,n
0,anthropic,claude-sonnet-4-6,aut_automobile_tire,g2_css_static3,diverge,150
1,anthropic,claude-sonnet-4-6,aut_automobile_tire,g2_css_static3,vanilla,150
2,anthropic,claude-sonnet-4-6,aut_automobile_tire,simplestrat5,diverge,150
3,anthropic,claude-sonnet-4-6,aut_automobile_tire,simplestrat5,vanilla,150
4,anthropic,claude-sonnet-4-6,aut_button,g2_css_static3,diverge,150
5,anthropic,claude-sonnet-4-6,aut_button,g2_css_static3,vanilla,150
6,anthropic,claude-sonnet-4-6,aut_button,simplestrat5,diverge,150
7,anthropic,claude-sonnet-4-6,aut_button,simplestrat5,vanilla,150
8,anthropic,claude-sonnet-4-6,aut_key,g2_css_static3,diverge,150
9,anthropic,claude-sonnet-4-6,aut_key,g2_css_static3,vanilla,150


In [14]:
aut_rows_csv = ANALYSIS_READY_AUT_DIR / "aut_claus_quality_rows_analysis_ready.csv"
aut_rows_pkl = ANALYSIS_READY_AUT_DIR / "aut_claus_quality_rows_analysis_ready.pkl"
aut_score_cols_json = ANALYSIS_READY_AUT_DIR / "aut_claus_candidate_score_columns.json"

aut_quality_rows.to_csv(aut_rows_csv, index=False)
aut_quality_rows.to_pickle(aut_rows_pkl)

with open(aut_score_cols_json, "w", encoding="utf-8") as f:
    json.dump({"candidate_score_columns": aut_score_cols}, f, indent=2)

aut_cell_summary = (
    aut_quality_rows
    .groupby(["provider", "provider_label", "model", "task_family", "task_id", "task_label", "method", "method_label", "strategy"], observed=True)
    .agg(
        n=("quality_row_id", "count"),
        mean_response_words=("response_word_count", "mean"),
        mean_response_chars=("response_n_chars", "mean"),
    )
    .reset_index()
)

for c in aut_score_cols:
    aut_cell_summary[f"mean__{c}"] = (
        aut_quality_rows
        .assign(_score=pd.to_numeric(aut_quality_rows[c], errors="coerce"))
        .groupby(["provider", "provider_label", "model", "task_family", "task_id", "task_label", "method", "method_label", "strategy"], observed=True)["_score"]
        .mean()
        .values
    )

aut_summary_csv = ANALYSIS_READY_AUT_DIR / "aut_claus_quality_cell_summary_analysis_ready.csv"
aut_cell_summary.to_csv(aut_summary_csv, index=False)

print("Saved AUT analysis-ready rows:")
print(aut_rows_csv)
print(aut_rows_pkl)

print("\nSaved AUT score column list:")
print(aut_score_cols_json)

print("\nSaved AUT cell summary:")
print(aut_summary_csv)

display(aut_cell_summary.head())

Saved AUT analysis-ready rows:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/AUT_CLAUS/aut_claus_quality_rows_analysis_ready.csv
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/AUT_CLAUS/aut_claus_quality_rows_analysis_ready.pkl

Saved AUT score column list:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/AUT_CLAUS/aut_claus_candidate_score_columns.json

Saved AUT cell summary:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/AUT_CLAUS/aut_claus_quality_cell_summary_analysis_ready.csv


,provider,provider_label,model,task_family,task_id,task_label,method,method_label,strategy,n,...,mean__cap_returned__round,mean__cap_returned__slot_num,mean__cap_returned__stratum_id,mean__cap_returned__anchor_count,mean__cap_returned__context_count,mean__cap_returned__response_word_count,mean__cap_returned__response_n_chars,mean__cap_returned__rough_sentence_count,mean__cap_returned__prediction,mean__cap_returned__cap_returned_row
0,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,aut,aut_automobile_tire,AUT: automobile tire,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",diverge,150,...,2.0,75.5,NaN,3.0,3.0,27.120000,168.813333,1.0,0.684200,75.5
1,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,aut,aut_automobile_tire,AUT: automobile tire,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",vanilla,150,...,2.0,75.5,NaN,3.0,3.0,24.820000,149.986667,1.0,0.665333,225.5
2,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,aut,aut_automobile_tire,AUT: automobile tire,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",diverge,150,...,2.0,75.5,3.0,0.0,1.0,29.040000,194.560000,1.0,0.710000,375.5
3,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,aut,aut_automobile_tire,AUT: automobile tire,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",vanilla,150,...,2.0,75.5,3.0,0.0,1.0,25.586667,169.053333,1.0,0.679067,525.5
4,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,aut,aut_button,AUT: button,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",diverge,150,...,2.0,75.5,NaN,3.0,3.0,19.640000,109.946667,1.0,0.661800,675.5


In [15]:
story_returned_files = list_returned_files(RETURNED_STORY_DIR)

print("Returned story files found:", len(story_returned_files))
for p in story_returned_files:
    print(" -", p.name)

if len(story_returned_files) == 0:
    raise RuntimeError(f"No returned story files found in {RETURNED_STORY_DIR}")

story_merged_parts = []

for returned_path in story_returned_files:
    merged = merge_returned_with_upload(
        returned_path=returned_path,
        upload_dir=STORY_UPLOAD_DIR,
    )
    merged["cap_tool"] = "MAoSS"
    merged["quality_task_family"] = "story"
    story_merged_parts.append(merged)

story_quality_rows = pd.concat(story_merged_parts, ignore_index=True, sort=False)

if story_quality_rows["quality_row_id"].duplicated().any():
    dupes = story_quality_rows[story_quality_rows["quality_row_id"].duplicated(keep=False)]
    display(dupes[["quality_row_id", "returned_file_name", "upload_file_name", "item", "response"]].head(50))
    raise RuntimeError("Duplicate story quality_row_id values after merging returned files.")

story_score_cols = candidate_score_columns(story_quality_rows)

print("Merged story rows:", len(story_quality_rows))
print("Candidate story score columns:")
print(story_score_cols)

display(story_quality_rows.head())
display(
    story_quality_rows
    .groupby(["provider", "model", "task_id", "method", "strategy"], observed=True)
    .size()
    .reset_index(name="n")
)

Returned story files found: 9
 - RETURNED__MAoSS_upload_story_anthropic_claude_sonnet_4_6_part01of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_anthropic_claude_sonnet_4_6_part02of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_anthropic_claude_sonnet_4_6_part03of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_gemini_gemini_2_5_pro_part01of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_gemini_gemini_2_5_pro_part02of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_gemini_gemini_2_5_pro_part03of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_openai_gpt_5_4_part01of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_openai_gpt_5_4_part02of03_FULL_with_ids.csv
 - RETURNED__MAoSS_upload_story_openai_gpt_5_4_part03of03_FULL_with_ids.csv
Merged story rows: 7200
Candidate story score columns:
['maoss_provider_upload_row', 'maoss_split_index', 'maoss_split_total', 'maoss_split_row', 'cap_returned__cap_upload_row', 'cap_returned__round', 'cap_returned__slot_num',

,item,response,quality_row_id,cap_upload_row,provider,provider_label,model,experiment_id,task_id,task_label,...,cap_returned__modelname,cap_returned__text,cap_returned__cap_returned_row,merge_method,returned_file_path,upload_file_path,upload_file_name,returned_file_name,cap_tool,quality_task_family
0,Write a short horror story designed to chill t...,"The voicemail was only four seconds long, and ...",1add549d4dd2c98cc5acc93b,3001,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,MAoSS,"the voicemail was only four seconds long, and ...",1,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,MAoSS_upload_story_anthropic_claude_sonnet_4_6...,RETURNED__MAoSS_upload_story_anthropic_claude_...,MAoSS,story
1,Write a short horror story designed to chill t...,"The voicemail was only twelve seconds long, le...",3ce88e6dcaf5e77c641b80c5,3004,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,MAoSS,"the voicemail was only twelve seconds long, le...",2,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,MAoSS_upload_story_anthropic_claude_sonnet_4_6...,RETURNED__MAoSS_upload_story_anthropic_claude_...,MAoSS,story
2,Write a short horror story designed to chill t...,"The summer I turned seventeen, my twin sister ...",fa996c0d354b9103f086f54a,3007,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,MAoSS,"the summer i turned seventeen, my twin sister ...",3,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,MAoSS_upload_story_anthropic_claude_sonnet_4_6...,RETURNED__MAoSS_upload_story_anthropic_claude_...,MAoSS,story
3,Write a short horror story designed to chill t...,"The summer my grandmother died, she made me pr...",5b4556b01a9cf5f0d63b9e9a,3010,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,MAoSS,"the summer my grandmother died, she made me pr...",4,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,MAoSS_upload_story_anthropic_claude_sonnet_4_6...,RETURNED__MAoSS_upload_story_anthropic_claude_...,MAoSS,story
4,Write a short horror story designed to chill t...,The voicemail from my grandmother lasted forty...,9a7e956272ecd2c43860af50,3013,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,anthropic_followup_simplestrat5_g2css3,story_horror,Story: horror,...,MAoSS,the voicemail from my grandmother lasted forty...,5,quality_row_id,analysis_outputs/deflect_creativity_followup_q...,analysis_outputs/deflect_creativity_followup_q...,MAoSS_upload_story_anthropic_claude_sonnet_4_6...,RETURNED__MAoSS_upload_story_anthropic_claude_...,MAoSS,story


,provider,model,task_id,method,strategy,n
0,anthropic,claude-sonnet-4-6,story_horror,g2_css_static3,diverge,150
1,anthropic,claude-sonnet-4-6,story_horror,g2_css_static3,vanilla,150
2,anthropic,claude-sonnet-4-6,story_horror,simplestrat5,diverge,150
3,anthropic,claude-sonnet-4-6,story_horror,simplestrat5,vanilla,150
4,anthropic,claude-sonnet-4-6,story_jungle,g2_css_static3,diverge,150
5,anthropic,claude-sonnet-4-6,story_jungle,g2_css_static3,vanilla,150
6,anthropic,claude-sonnet-4-6,story_jungle,simplestrat5,diverge,150
7,anthropic,claude-sonnet-4-6,story_jungle,simplestrat5,vanilla,150
8,anthropic,claude-sonnet-4-6,story_life_last_seconds,g2_css_static3,diverge,150
9,anthropic,claude-sonnet-4-6,story_life_last_seconds,g2_css_static3,vanilla,150


In [16]:
story_rows_csv = ANALYSIS_READY_STORY_DIR / "story_maoss_quality_rows_analysis_ready.csv"
story_rows_pkl = ANALYSIS_READY_STORY_DIR / "story_maoss_quality_rows_analysis_ready.pkl"
story_score_cols_json = ANALYSIS_READY_STORY_DIR / "story_maoss_candidate_score_columns.json"

story_quality_rows.to_csv(story_rows_csv, index=False)
story_quality_rows.to_pickle(story_rows_pkl)

with open(story_score_cols_json, "w", encoding="utf-8") as f:
    json.dump({"candidate_score_columns": story_score_cols}, f, indent=2)

story_cell_summary = (
    story_quality_rows
    .groupby(["provider", "provider_label", "model", "task_family", "task_id", "task_label", "method", "method_label", "strategy"], observed=True)
    .agg(
        n=("quality_row_id", "count"),
        mean_response_words=("response_word_count", "mean"),
        mean_response_chars=("response_n_chars", "mean"),
        mean_sentence_count=("rough_sentence_count", "mean"),
    )
    .reset_index()
)

for c in story_score_cols:
    story_cell_summary[f"mean__{c}"] = (
        story_quality_rows
        .assign(_score=pd.to_numeric(story_quality_rows[c], errors="coerce"))
        .groupby(["provider", "provider_label", "model", "task_family", "task_id", "task_label", "method", "method_label", "strategy"], observed=True)["_score"]
        .mean()
        .values
    )

story_summary_csv = ANALYSIS_READY_STORY_DIR / "story_maoss_quality_cell_summary_analysis_ready.csv"
story_cell_summary.to_csv(story_summary_csv, index=False)

print("Saved story analysis-ready rows:")
print(story_rows_csv)
print(story_rows_pkl)

print("\nSaved story score column list:")
print(story_score_cols_json)

print("\nSaved story cell summary:")
print(story_summary_csv)

display(story_cell_summary.head())

Saved story analysis-ready rows:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/story_MAoSS/story_maoss_quality_rows_analysis_ready.csv
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/story_MAoSS/story_maoss_quality_rows_analysis_ready.pkl

Saved story score column list:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/story_MAoSS/story_maoss_candidate_score_columns.json

Saved story cell summary:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/story_MAoSS/story_maoss_quality_cell_summary_analysis_ready.csv


,provider,provider_label,model,task_family,task_id,task_label,method,method_label,strategy,n,...,mean__cap_returned__context_count,mean__cap_returned__response_word_count,mean__cap_returned__response_n_chars,mean__cap_returned__rough_sentence_count,mean__cap_returned__maoss_provider_upload_row,mean__cap_returned__maoss_split_index,mean__cap_returned__maoss_split_total,mean__cap_returned__maoss_split_row,mean__cap_returned__prediction,mean__cap_returned__cap_returned_row
0,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,story,story_horror,Story: horror,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",diverge,150,...,3.0,223.313333,1223.880000,7.940000,75.5,2.0,3.0,25.5,0.659000,25.5
1,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,story,story_horror,Story: horror,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",vanilla,150,...,3.0,222.440000,1210.780000,7.940000,225.5,2.0,3.0,75.5,0.668533,75.5
2,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,story,story_horror,Story: horror,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",diverge,150,...,1.0,225.446667,1267.246667,7.913333,375.5,2.0,3.0,125.5,0.671200,125.5
3,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,story,story_horror,Story: horror,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",vanilla,150,...,1.0,218.300000,1237.640000,7.793333,525.5,2.0,3.0,175.5,0.669200,175.5
4,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,story,story_jungle,Story: jungle,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",diverge,150,...,3.0,264.386667,1559.840000,7.986667,675.5,2.0,3.0,225.5,0.705467,225.5


In [17]:
analysis_ready_manifest = {
    "quality_run_id": QUALITY_RUN_ID,
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "quality_output_root": str(QUALITY_OUTPUT_ROOT),
    "returned_dir": str(RETURNED_DIR),
    "returned_aut_dir": str(RETURNED_AUT_DIR),
    "returned_story_dir": str(RETURNED_STORY_DIR),
    "analysis_ready_dir": str(ANALYSIS_READY_DIR),
    "aut": {
        "returned_files": [str(p) for p in aut_returned_files],
        "rows_csv": str(aut_rows_csv),
        "rows_pkl": str(aut_rows_pkl),
        "cell_summary_csv": str(aut_summary_csv),
        "candidate_score_columns_json": str(aut_score_cols_json),
        "candidate_score_columns": aut_score_cols,
        "n_rows": int(len(aut_quality_rows)),
    },
    "story": {
        "returned_files": [str(p) for p in story_returned_files],
        "rows_csv": str(story_rows_csv),
        "rows_pkl": str(story_rows_pkl),
        "cell_summary_csv": str(story_summary_csv),
        "candidate_score_columns_json": str(story_score_cols_json),
        "candidate_score_columns": story_score_cols,
        "n_rows": int(len(story_quality_rows)),
    },
    "merge_notes": [
        "Preferred merge key is quality_row_id when preserved by CAP.",
        "If quality_row_id is absent from the returned file, merge uses row order against the original upload file.",
        "Original upload filenames are inferred by stripping RETURNED__ from returned filenames.",
        "All returned columns are preserved, usually prefixed with cap_returned__.",
        "Candidate score columns are detected heuristically and saved separately for inspection.",
    ],
}

analysis_ready_manifest_path = ANALYSIS_READY_DIR / "analysis_ready_quality_manifest_aut_story.json"

with open(analysis_ready_manifest_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(analysis_ready_manifest), f, indent=2, ensure_ascii=False)

print("Saved analysis-ready manifest:")
print(analysis_ready_manifest_path)

print("\nMain files for later analysis notebooks:")
print(aut_rows_pkl)
print(story_rows_pkl)

Saved analysis-ready manifest:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/analysis_ready_quality_manifest_aut_story.json

Main files for later analysis notebooks:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/AUT_CLAUS/aut_claus_quality_rows_analysis_ready.pkl
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/story_MAoSS/story_maoss_quality_rows_analysis_ready.pkl


In [18]:
# -------------------------------------------------------------------
# Slogan lexical quality export
# -------------------------------------------------------------------

SLOGAN_ANALYSIS_READY_DIR = ANALYSIS_READY_DIR / "slogan_lexical_non_template"
SLOGAN_ANALYSIS_READY_DIR.mkdir(parents=True, exist_ok=True)

print("Slogan analysis-ready outputs will be written here:")
print(SLOGAN_ANALYSIS_READY_DIR)

Slogan analysis-ready outputs will be written here:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template


In [19]:
from collections import Counter

def normalize_slogan_for_lexical_metric(text: Any) -> str:
    """
    Normalize slogan text for lexical-template diagnostics.
    Keeps words/numbers/apostrophes; removes punctuation/casing differences.
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = text.replace("’", "'")
    text = re.sub(r"[^a-z0-9'\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def slogan_tokens(text: Any) -> list[str]:
    norm = normalize_slogan_for_lexical_metric(text)
    return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", norm)


def make_ngrams(tokens: list[str], ns: tuple[int, ...] = (2, 3)) -> list[str]:
    out = []
    for n in ns:
        if len(tokens) >= n:
            out.extend([" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)])
    return out


def compute_leave_one_out_ngram_commonness(
    df: pd.DataFrame,
    task_col: str = "task_id",
    text_col: str = "slogan_text",
) -> pd.DataFrame:
    """
    For each slogan, compute average leave-one-out document-frequency proportion
    of its unique bigrams/trigrams within the same task.

    Higher raw commonness = more reliance on repeated phrase templates.
    The final quality metric reverses and z-scores this within task:
        slogan_lexical_non_template_z_task = -z(commonness)
    """
    out = df.copy()

    out["slogan_norm"] = out[text_col].map(normalize_slogan_for_lexical_metric)
    out["slogan_tokens"] = out[text_col].map(slogan_tokens)
    out["slogan_bigram_trigram_list"] = out["slogan_tokens"].map(lambda toks: make_ngrams(toks, ns=(2, 3)))
    out["slogan_bigram_trigram_unique"] = out["slogan_bigram_trigram_list"].map(lambda xs: sorted(set(xs)))
    out["slogan_n_bigram_trigram_unique"] = out["slogan_bigram_trigram_unique"].map(len)

    commonness = pd.Series(index=out.index, dtype=float)

    for task_id, g in out.groupby(task_col, observed=True):
        idx = g.index.to_list()
        n_docs = len(g)

        doc_freq = Counter()
        for grams in g["slogan_bigram_trigram_unique"]:
            doc_freq.update(grams)

        denom = max(n_docs - 1, 1)

        for row_idx, grams in zip(g.index, g["slogan_bigram_trigram_unique"]):
            if len(grams) == 0:
                commonness.loc[row_idx] = 0.0
                continue

            # LOO proportion: how often this row's phrase templates appear in other rows.
            vals = [(doc_freq[ng] - 1) / denom for ng in grams]
            commonness.loc[row_idx] = float(np.mean(vals))

    out["slogan_lexical_boilerplate_commonness_loo"] = commonness.astype(float)

    # Within-task z-score, then reverse so higher = less template-like.
    out["slogan_lexical_commonness_z_task"] = np.nan
    out["slogan_lexical_non_template_z_task"] = np.nan

    for task_id, g in out.groupby(task_col, observed=True):
        vals = pd.to_numeric(g["slogan_lexical_boilerplate_commonness_loo"], errors="coerce")
        mu = float(vals.mean())
        sd = float(vals.std(ddof=0))

        if not np.isfinite(sd) or sd <= 0:
            z = pd.Series(np.zeros(len(g)), index=g.index, dtype=float)
        else:
            z = (vals - mu) / sd

        out.loc[g.index, "slogan_lexical_commonness_z_task"] = z
        out.loc[g.index, "slogan_lexical_non_template_z_task"] = -z

    return out


def add_slogan_exact_duplicate_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    cell_keys = ["provider", "model", "task_id", "method", "strategy"]

    out["slogan_exact_count_within_cell"] = (
        out
        .groupby(cell_keys + ["slogan_norm"], observed=True)["quality_row_id"]
        .transform("count")
    )

    out["slogan_has_exact_duplicate_within_cell"] = out["slogan_exact_count_within_cell"].gt(1)

    cell_n = out.groupby(cell_keys, observed=True)["quality_row_id"].transform("count")
    out["slogan_exact_duplicate_rate_within_cell_loo"] = np.where(
        cell_n.gt(1),
        (out["slogan_exact_count_within_cell"] - 1) / (cell_n - 1),
        0.0,
    )

    return out

In [20]:
# Use the normalized `ideas` table created earlier in this notebook.
# It contains all completed follow-up provider outputs currently available.

if "ideas" not in globals():
    raise RuntimeError("Expected `ideas` from the earlier normalization cell. Rerun the notebook from the load/normalize cells.")

slogan_rows = ideas[
    ideas["is_success"]
    & ideas["task_family"].eq("slogan")
].copy()

if slogan_rows.empty:
    raise RuntimeError("No successful slogan rows found in `ideas`.")

slogan_rows["slogan_text"] = slogan_rows["response"]

# Ensure stable labels.
slogan_rows["task_family_label"] = slogan_rows["task_family"].map(TASK_FAMILY_LABEL).fillna("Slogan")
slogan_rows["task_label"] = slogan_rows["task_id"].map(TASK_LABEL).fillna(slogan_rows.get("task_label", slogan_rows["task_id"]))
slogan_rows["provider_label"] = slogan_rows["provider"].map(PROVIDER_LABEL).fillna(slogan_rows["provider"])

slogan_lexical_rows = compute_leave_one_out_ngram_commonness(
    slogan_rows,
    task_col="task_id",
    text_col="slogan_text",
)

slogan_lexical_rows = add_slogan_exact_duplicate_diagnostics(slogan_lexical_rows)

# Keep the main metric numeric and nonmissing.
slogan_lexical_rows["slogan_lexical_non_template_z_task"] = pd.to_numeric(
    slogan_lexical_rows["slogan_lexical_non_template_z_task"],
    errors="coerce",
)

missing_metric = slogan_lexical_rows["slogan_lexical_non_template_z_task"].isna().sum()
if missing_metric > 0:
    display(
        slogan_lexical_rows.loc[
            slogan_lexical_rows["slogan_lexical_non_template_z_task"].isna(),
            ["provider", "model", "task_id", "method", "strategy", "slogan_text"]
        ].head(50)
    )
    raise RuntimeError(f"{missing_metric} slogan rows have missing lexical non-template scores.")

print("Slogan lexical rows:", slogan_lexical_rows.shape)

display(
    slogan_lexical_rows
    .groupby(["provider", "model", "task_id", "method", "strategy"], observed=True)
    .agg(
        n=("quality_row_id", "count"),
        mean_non_template=("slogan_lexical_non_template_z_task", "mean"),
        sd_non_template=("slogan_lexical_non_template_z_task", "std"),
        mean_boilerplate_commonness=("slogan_lexical_boilerplate_commonness_loo", "mean"),
        exact_duplicate_rate=("slogan_has_exact_duplicate_within_cell", "mean"),
    )
    .reset_index()
)

Slogan lexical rows: (5400, 76)


,provider,model,task_id,method,strategy,n,mean_non_template,sd_non_template,mean_boilerplate_commonness,exact_duplicate_rate
0,anthropic,claude-sonnet-4-6,slogan_blood_donation,g2_css_static3,diverge,150,-0.468494,0.866274,0.024585,0.820000
1,anthropic,claude-sonnet-4-6,slogan_blood_donation,g2_css_static3,vanilla,150,-0.971621,1.095905,0.032165,0.886667
2,anthropic,claude-sonnet-4-6,slogan_blood_donation,simplestrat5,diverge,150,0.418738,0.577062,0.011220,0.773333
3,anthropic,claude-sonnet-4-6,slogan_blood_donation,simplestrat5,vanilla,150,0.553618,0.403362,0.009188,0.726667
4,anthropic,claude-sonnet-4-6,slogan_smartphone,g2_css_static3,diverge,150,-1.146461,0.978730,0.041167,0.926667
5,anthropic,claude-sonnet-4-6,slogan_smartphone,g2_css_static3,vanilla,150,-0.242809,1.323347,0.022794,0.713333
6,anthropic,claude-sonnet-4-6,slogan_smartphone,simplestrat5,diverge,150,0.517969,0.333008,0.007327,0.733333
7,anthropic,claude-sonnet-4-6,slogan_smartphone,simplestrat5,vanilla,150,0.431355,0.311308,0.009088,0.826667
8,anthropic,claude-sonnet-4-6,slogan_soda,g2_css_static3,diverge,150,-0.403352,1.249563,0.017983,0.753333
9,anthropic,claude-sonnet-4-6,slogan_soda,g2_css_static3,vanilla,150,-0.097430,0.813699,0.014698,0.666667


In [21]:
SLOGAN_MAIN_METRIC = "slogan_lexical_non_template_z_task"

SLOGAN_METRIC_DEFINITIONS = {
    SLOGAN_MAIN_METRIC: {
        "metric_family": "slogan_lexical",
        "metric_label": "Slogan: non-template",
        "metric_type": "continuous_z",
        "higher_is_better": True,
        "description": (
            "Negative within-task z-score of leave-one-out bigram/trigram "
            "boilerplate commonness. Higher values indicate less reliance on repeated lexical templates."
        ),
    }
}

slogan_keep_cols = [
    "quality_row_id",
    "provider",
    "provider_label",
    "model",
    "experiment_id",
    "task_id",
    "task_label",
    "task_family",
    "task_family_label",
    "method",
    "method_label",
    "condition",
    "strategy",
    "round",
    "slot_id",
    "slot_num",
    "stratum_id",
    "anchor_count",
    "context_count",
    "request_key",
    "status",
    "slogan_text",
    "slogan_norm",
    "response_word_count",
    "response_n_chars",
    "slogan_n_bigram_trigram_unique",
    "slogan_lexical_boilerplate_commonness_loo",
    "slogan_lexical_commonness_z_task",
    "slogan_lexical_non_template_z_task",
    "slogan_exact_count_within_cell",
    "slogan_has_exact_duplicate_within_cell",
    "slogan_exact_duplicate_rate_within_cell_loo",
    "usage_prompt_token_count",
    "usage_candidates_token_count",
    "usage_thoughts_token_count",
    "usage_total_token_count",
    "usage_input_tokens",
    "usage_output_tokens",
    "usage_cached_tokens",
    "usage_reasoning_tokens",
    "batch_id",
    "batch_name",
    "batch_custom_id",
    "batch_output_file",
    "source_output_path",
]

slogan_keep_cols = [c for c in slogan_keep_cols if c in slogan_lexical_rows.columns]

slogan_quality_rows_analysis_ready = (
    slogan_lexical_rows[slogan_keep_cols]
    .sort_values(["provider", "model", "task_id", "method", "strategy", "slot_num", "request_key"])
    .reset_index(drop=True)
)

slogan_quality_long_analysis_ready = slogan_quality_rows_analysis_ready[
    [
        "quality_row_id",
        "provider",
        "provider_label",
        "model",
        "experiment_id",
        "task_id",
        "task_label",
        "task_family",
        "task_family_label",
        "method",
        "method_label",
        "condition",
        "strategy",
        "round",
        "slot_id",
        "slot_num",
        "anchor_count",
        "context_count",
        "request_key",
        "slogan_text",
        SLOGAN_MAIN_METRIC,
    ]
].copy()

slogan_quality_long_analysis_ready = slogan_quality_long_analysis_ready.rename(
    columns={SLOGAN_MAIN_METRIC: "value"}
)

slogan_quality_long_analysis_ready["metric"] = SLOGAN_MAIN_METRIC
slogan_quality_long_analysis_ready["metric_label"] = SLOGAN_METRIC_DEFINITIONS[SLOGAN_MAIN_METRIC]["metric_label"]
slogan_quality_long_analysis_ready["outcome_domain"] = "quality"
slogan_quality_long_analysis_ready["analysis_scale"] = "within-task z"
slogan_quality_long_analysis_ready["higher_is_better"] = True

slogan_cell_summary_analysis_ready = (
    slogan_quality_rows_analysis_ready
    .groupby(
        [
            "provider",
            "provider_label",
            "model",
            "task_family",
            "task_family_label",
            "task_id",
            "task_label",
            "method",
            "method_label",
            "condition",
            "strategy",
            "round",
        ],
        observed=True,
    )
    .agg(
        n=("quality_row_id", "count"),
        value=(SLOGAN_MAIN_METRIC, "mean"),
        sd_value=(SLOGAN_MAIN_METRIC, "std"),
        se_value=(SLOGAN_MAIN_METRIC, lambda x: float(np.std(x, ddof=1) / np.sqrt(len(x))) if len(x) > 1 else np.nan),
        mean_boilerplate_commonness=("slogan_lexical_boilerplate_commonness_loo", "mean"),
        mean_response_words=("response_word_count", "mean"),
        mean_response_chars=("response_n_chars", "mean"),
        exact_duplicate_rate=("slogan_has_exact_duplicate_within_cell", "mean"),
        unique_slogan_rate=("slogan_norm", lambda x: x.nunique() / len(x) if len(x) else np.nan),
    )
    .reset_index()
)

slogan_cell_summary_analysis_ready["metric"] = SLOGAN_MAIN_METRIC
slogan_cell_summary_analysis_ready["metric_label"] = SLOGAN_METRIC_DEFINITIONS[SLOGAN_MAIN_METRIC]["metric_label"]
slogan_cell_summary_analysis_ready["outcome_domain"] = "quality"
slogan_cell_summary_analysis_ready["analysis_scale"] = "within-task z"
slogan_cell_summary_analysis_ready["higher_is_better"] = True

# Save files.
slogan_rows_csv = SLOGAN_ANALYSIS_READY_DIR / "slogan_lexical_quality_rows_analysis_ready.csv"
slogan_rows_pkl = SLOGAN_ANALYSIS_READY_DIR / "slogan_lexical_quality_rows_analysis_ready.pkl"

slogan_long_csv = SLOGAN_ANALYSIS_READY_DIR / "slogan_lexical_quality_long_analysis_ready.csv"
slogan_long_pkl = SLOGAN_ANALYSIS_READY_DIR / "slogan_lexical_quality_long_analysis_ready.pkl"

slogan_summary_csv = SLOGAN_ANALYSIS_READY_DIR / "slogan_lexical_quality_cell_summary_analysis_ready.csv"

slogan_metric_defs_json = SLOGAN_ANALYSIS_READY_DIR / "slogan_lexical_metric_definitions.json"

slogan_quality_rows_analysis_ready.to_csv(slogan_rows_csv, index=False)
slogan_quality_rows_analysis_ready.to_pickle(slogan_rows_pkl)

slogan_quality_long_analysis_ready.to_csv(slogan_long_csv, index=False)
slogan_quality_long_analysis_ready.to_pickle(slogan_long_pkl)

slogan_cell_summary_analysis_ready.to_csv(slogan_summary_csv, index=False)

with open(slogan_metric_defs_json, "w", encoding="utf-8") as f:
    json.dump(SLOGAN_METRIC_DEFINITIONS, f, indent=2, ensure_ascii=False)

print("Saved slogan row-level files:")
print(slogan_rows_csv)
print(slogan_rows_pkl)

print("\nSaved slogan long-format files:")
print(slogan_long_csv)
print(slogan_long_pkl)

print("\nSaved slogan cell summary:")
print(slogan_summary_csv)

print("\nSaved slogan metric definitions:")
print(slogan_metric_defs_json)

display(slogan_cell_summary_analysis_ready.head())

Saved slogan row-level files:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_rows_analysis_ready.csv
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_rows_analysis_ready.pkl

Saved slogan long-format files:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_long_analysis_ready.csv
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_long_analysis_ready.pkl

Saved slogan cell summary:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_cell_summary_analysis_r

,provider,provider_label,model,task_family,task_family_label,task_id,task_label,method,method_label,condition,...,mean_boilerplate_commonness,mean_response_words,mean_response_chars,exact_duplicate_rate,unique_slogan_rate,metric,metric_label,outcome_domain,analysis_scale,higher_is_better
0,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,slogan,Slogan,slogan_blood_donation,Blood donation slogan,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",g2_css_static3,...,0.024585,5.780000,34.500000,0.820000,0.360000,slogan_lexical_non_template_z_task,Slogan: non-template,quality,within-task z,True
1,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,slogan,Slogan,slogan_blood_donation,Blood donation slogan,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",g2_css_static3,...,0.032165,5.813333,28.153333,0.886667,0.220000,slogan_lexical_non_template_z_task,Slogan: non-template,quality,within-task z,True
2,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,slogan,Slogan,slogan_blood_donation,Blood donation slogan,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",simplestrat5,...,0.011220,5.826667,35.293333,0.773333,0.386667,slogan_lexical_non_template_z_task,Slogan: non-template,quality,within-task z,True
3,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,slogan,Slogan,slogan_blood_donation,Blood donation slogan,simplestrat5,"SimpleStrat-lite, 5 fixed auto-stratified sema...",simplestrat5,...,0.009188,5.726667,35.253333,0.726667,0.453333,slogan_lexical_non_template_z_task,Slogan: non-template,quality,within-task z,True
4,anthropic,Claude Sonnet 4.6,claude-sonnet-4-6,slogan,Slogan,slogan_smartphone,Smartphone slogan,g2_css_static3,"G2-inspired static CSS, 3 representative R1 ex...",g2_css_static3,...,0.041167,5.526667,34.613333,0.926667,0.186667,slogan_lexical_non_template_z_task,Slogan: non-template,quality,within-task z,True


In [22]:
slogan_analysis_ready_manifest = {
    "quality_run_id": QUALITY_RUN_ID,
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "quality_output_root": str(QUALITY_OUTPUT_ROOT),
    "analysis_ready_dir": str(ANALYSIS_READY_DIR),
    "slogan_analysis_ready_dir": str(SLOGAN_ANALYSIS_READY_DIR),
    "metric": SLOGAN_MAIN_METRIC,
    "metric_definition": SLOGAN_METRIC_DEFINITIONS[SLOGAN_MAIN_METRIC],
    "rows_csv": str(slogan_rows_csv),
    "rows_pkl": str(slogan_rows_pkl),
    "long_csv": str(slogan_long_csv),
    "long_pkl": str(slogan_long_pkl),
    "cell_summary_csv": str(slogan_summary_csv),
    "metric_definitions_json": str(slogan_metric_defs_json),
    "n_rows": int(len(slogan_quality_rows_analysis_ready)),
    "included_providers": sorted(slogan_quality_rows_analysis_ready["provider"].unique().tolist()),
    "included_tasks": sorted(slogan_quality_rows_analysis_ready["task_id"].unique().tolist()),
    "included_methods": sorted(slogan_quality_rows_analysis_ready["method"].unique().tolist()),
    "included_strategies": sorted(slogan_quality_rows_analysis_ready["strategy"].unique().tolist()),
    "notes": [
        "Only the lexical slogan quality metric is exported as an analysis metric.",
        "The main metric is slogan_lexical_non_template_z_task.",
        "Higher values indicate less reliance on repeated bigram/trigram phrase templates.",
        "Exact-duplicate diagnostics are included for audit but are not marked as primary quality metrics.",
        "Rerun from the load/normalize cells after Anthropic finishes to include Anthropic automatically.",
    ],
}

slogan_manifest_path = SLOGAN_ANALYSIS_READY_DIR / "slogan_lexical_analysis_ready_manifest.json"

with open(slogan_manifest_path, "w", encoding="utf-8") as f:
    json.dump(json_safe(slogan_analysis_ready_manifest), f, indent=2, ensure_ascii=False)

print("Saved slogan analysis-ready manifest:")
print(slogan_manifest_path)

print("\nMain files for later analysis notebooks:")
print(slogan_rows_pkl)
print(slogan_long_pkl)
print(slogan_summary_csv)

Saved slogan analysis-ready manifest:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_analysis_ready_manifest.json

Main files for later analysis notebooks:
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_rows_analysis_ready.pkl
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_long_analysis_ready.pkl
analysis_outputs/deflect_creativity_followup_quality/quality_20260523_141333/analysis_ready_quality_scores/slogan_lexical_non_template/slogan_lexical_quality_cell_summary_analysis_ready.csv
